# 31_pooledB — 10% pooled Stage-B replay ablation

Compares the exact Notebook 31 model against one parameter-identical variant. During each dataset expert's Stage-B warm-start, every owned batch keeps all of its original rows and receives a deterministic, class-balanced pooled replay term with weight 10%. The baseline Stage-A checkpoint is reused; Stage B and Stage C are retrained.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER = 'selimsidan'
GITHUB_REPO = 'dataset_moe_nids'
GITHUB_BRANCH = 'main'
GITHUB_SECRET_NAME = 'GITHUB_TOKEN'
DRIVE_DATA_DIR = '/content/drive/MyDrive/NIDS_datasets'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs'
STUDIES = [
    ('baseline', 'config/compact_soft_moe_3seed.yaml', 'nfv3_4way_moe_soft_comparable_repro_v1'),
    ('pooled_b10', 'config/compact_soft_moe_pooled_b10_3seed.yaml', 'nfv3_4way_moe_soft_pooledb10_v1'),
]
EXECUTE = False
MAX_NEW_SEEDS_PER_STUDY = 3
RUN_TESTS = True
# ==================================================================

## Secure checkout and environment setup

In [ ]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get(GITHUB_SECRET_NAME)
if not token: raise RuntimeError(f'Add {GITHUB_SECRET_NAME} in Colab Secrets and grant access.')
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git_env = os.environ | {'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader', 'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {auth}'}
repo = Path('/content') / GITHUB_REPO; url = f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'
if (repo / '.git').is_dir(): subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', GITHUB_BRANCH], env=git_env, check=True)
else: subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, '--single-branch', url, str(repo)], env=git_env, check=True)
git_env.clear(); token = auth = None; os.chdir(repo)
os.environ['NIDS_DRIVE_BASE'] = DRIVE_DATA_DIR; os.environ['NIDS_OUTPUT_DIR'] = DRIVE_OUTPUT_DIR
os.environ['NIDS_SCRATCH_DIR'] = '/content/dataset_moe_nids_scratch'; os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## Dry-run and resumable execution

The baseline is completed first if needed. Replay reservoirs are sampled deterministically within class×dataset strata, capped at 4,096 rows per stratum. The replay loss uses pooled class weights; checkpoint summaries record owned and replay exposure separately.

In [ ]:
import torch, yaml
from training.logging_utils import run_streaming_logged
if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime before executing the study.')
if RUN_TESTS:
    subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_compact_soft_moe_run.py', 'tests/test_notebook31_ablation_methods.py', 'tests/test_out_of_core.py', 'tests/test_training_observability.py'], check=True)
for condition, study_config, prefix in STUDIES:
    study = yaml.safe_load(Path(study_config).read_text())
    summary_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_summary'; summary_dir.mkdir(parents=True, exist_ok=True)
    replay = study['backbone']['training']['stage_b'].get('replay_fraction', 0.0)
    print(f'=== {condition}: replay_fraction={replay:.2f}, parameters={study["expected_total_parameters"]:,} ===')
    command = [sys.executable, '-u', '-m', 'training.compact_soft_moe_run', '--study-config', study_config, '--prefix', prefix]
    run_streaming_logged(command, str(summary_dir / 'Notebook_Dry_Run.log'), env=os.environ.copy(), heartbeat_seconds=30, label=f'31-pooledB-{condition}-dry-run')
    if EXECUTE:
        run_streaming_logged([*command, '--execute', '--max-new-seeds', str(MAX_NEW_SEEDS_PER_STUDY)], str(summary_dir / 'Notebook_Training_Stream.log'), env=os.environ.copy(), heartbeat_seconds=30, label=f'31-pooledB-{condition}-training')
if not EXECUTE: print('NO TRAINING WAS STARTED. Review the graph, set EXECUTE=True, and rerun this cell.')

## Paired performance and replay-accounting comparison

In [ ]:
import pandas as pd
from IPython.display import Image, display
summaries = {condition: Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_summary' for condition, _, prefix in STUDIES}
complete = {name: (path / 'Final_Study_Manifest.json').is_file() for name, path in summaries.items()}; print('Completion:', complete)
if all(complete.values()):
    validation = []; test = []; classes = []; histories = []; resources = []
    for name, path in summaries.items():
        for filename, target in [('Three_Seed_Guardrails.csv', validation), ('Three_Seed_Overall.csv', test), ('Three_Seed_Per_Class.csv', classes), ('Training_History_3Seed.csv', histories), ('Three_Seed_Resource_Accounting.csv', resources)]:
            frame = pd.read_csv(path / filename); frame.insert(0, 'condition', name); target.append(frame)
    validation = pd.concat(validation, ignore_index=True); test = pd.concat(test, ignore_index=True)
    display(validation[['condition', 'seed', 'validation_macro_f1', 'worst_dataset_macro_f1', 'rare_recall_mean']])
    pivot = validation.pivot(index='seed', columns='condition', values='validation_macro_f1'); pivot['delta_pooled_minus_baseline'] = pivot['pooled_b10'] - pivot['baseline']; display(pivot)
    metrics = [c for c in ['accuracy', 'macro_f1', 'macro_precision', 'macro_recall', 'weighted_f1', 'roc_auc_ovr_macro', 'pr_auc_ovr_macro'] if c in test]
    display(test.groupby('condition')[metrics].agg(['mean', 'std']))
    print('=== Per-class F1 mean ==='); display(pd.concat(classes, ignore_index=True).pivot_table(index='class', columns='condition', values='f1', aggfunc='mean'))
    stage_b = pd.concat(histories, ignore_index=True).query("stage == 'B'")
    replay_columns = [c for c in ['condition', 'seed', 'dataset', 'epoch', 'rows', 'replay_rows', 'train_total_loss', 'train_owned_ce_loss', 'train_replay_ce_loss'] if c in stage_b]
    print('=== Stage-B replay exposure and losses ==='); display(stage_b[replay_columns])
    print('=== Resource accounting ==='); display(pd.concat(resources, ignore_index=True))
else: print('Comparison appears after both three-seed manifests are complete.')

## Latent-space and training curves

In [ ]:
if all(complete.values()):
    for name, path in summaries.items():
        print('===', name, 'latent summary ==='); display(pd.read_csv(path / 'Latent_Snapshot_Summary_3Seed.csv'))
        for figure in sorted((path / 'training_figures').glob('*.png')): display(Image(filename=str(figure)))